In [1]:
from datetime import datetime
import json

In [2]:
# 1. INPUT DATA JSON DARI MODUL 2
candidate_json = {
    "metadata_dokumen": {
        "jenis_dokumen": "KTP",
        "nomor_dokumen": "3171012304980001",
    },
    "data_peserta": {
        "nama": "BUDI SANTOSO",
        "nik": "3171012304980001",
        "tanggal_lahir": "1998-04-23",
        "alamat": "JL MERDEKANO12",
        "pendidikan_terakhir": "SMK TEKNIK MESIN",
    },
    "status_kelengkapan": {"is_lengkap": True, "field_kosong": []},
    "kualitas_ekstraksi": {
        "confidence_score": 0.90,
        "status_flag": "VALID",
        "catatan_ambiguitas": [
            "Koreksi NIK: OCR membaca 17 digit (terdapat karakter '1' tambahan di awal). NIK dinormalisasi menjadi 16 digit.",
            "Koreksi Alamat: 'ILMERDEKANO12' dikoreksi menjadi 'JL MERDEKANO12'.",
        ],
    },
}

In [3]:
# 2. FUNGSI EVALUASI ATURAN BISNIS
def evaluate_business_rules(data_json):
    data_peserta = data_json.get("data_peserta", {})
    status_kelengkapan = data_json.get("status_kelengkapan", {})
    kualitas = data_json.get("kualitas_ekstraksi", {})

    flags = []
    status_verifikasi = "APPROVED"

    # --- RULE 1: KELENGKAPAN DOKUMEN ---
    if not status_kelengkapan.get("is_lengkap", False):
        flags.append("RED FLAG: Dokumen tidak lengkap / ada field kosong.")
        status_verifikasi = "NEEDS_HUMAN_REVIEW"

    # --- RULE 2: HITUNG & CEK SYARAT USIA (18 - 26 TAHUN) ---
    tgl_lahir_str = data_peserta.get("tanggal_lahir")
    usia = None

    if tgl_lahir_str:
        try:
            tgl_lahir = datetime.strptime(tgl_lahir_str, "%Y-%m-%d")
            today = datetime.now()
            # Hitung usia persis berdasarkan tahun, bulan, hari
            usia = (
                today.year
                - tgl_lahir.year
                - ((today.month, today.day) < (tgl_lahir.month, tgl_lahir.day))
            )

            # Validasi Rentang Usia Magang
            if usia < 18 or usia > 26:
                flags.append(
                    f"REJECT: Usia pendaftar ({usia} tahun) di luar rentang syarat magang (18-26 tahun)."
                )
                status_verifikasi = "REJECTED"
        except ValueError:
            flags.append("WARNING: Format tanggal lahir tidak valid.")
            status_verifikasi = "NEEDS_HUMAN_REVIEW"

    # --- RULE 3: CEK SYARAT MINIMAL PENDIDIKAN (SMA/SMK) ---
    pendidikan = data_peserta.get("pendidikan_terakhir", "").upper()
    allowed_edu = ["SMA", "SMK", "D3", "S1", "S2"]

    if not any(edu in pendidikan for edu in allowed_edu):
        flags.append(
            f"REJECT: Jenjang pendidikan ({pendidikan}) tidak memenuhi syarat minimal SMA/SMK."
        )
        status_verifikasi = "REJECTED"

    # --- RULE 4: EVALUASI CONFIDENCE SCORE & AMBIGUITAS ---
    confidence = kualitas.get("confidence_score", 1.0)
    if confidence < 0.85 and status_verifikasi != "REJECTED":
        flags.append(
            f"WARNING: Confidence score rendah ({confidence}). Perlu pengecekan manual."
        )
        status_verifikasi = "NEEDS_HUMAN_REVIEW"

    # Jika ada catatan koreksi dari LLM, masukkan ke log review admin
    catatan_ocr = kualitas.get("catatan_ambiguitas", [])
    if catatan_ocr:
        for note in catatan_ocr:
            flags.append(f"INFO SYSTEM: {note}")

    # --- OUTPUT HASIL KEPUTUSAN SISTEM ---
    result_summary = {
        "nama_pendaftar": data_peserta.get("nama"),
        "nik": data_peserta.get("nik"),
        "usia_pendaftar": f"{usia} Tahun" if usia else "N/A",
        "pendidikan": pendidikan,
        "status_akhir": status_verifikasi,
        "log_catatan_sistem": flags,
    }

    return result_summary

In [4]:
# 3. JALANKAN RULE ENGINE
print("==================================================")
print("     MODUL 3: AUTOMATED BUSINESS RULE ENGINE      ")
print("==================================================")
evaluation_result = evaluate_business_rules(candidate_json)

print(json.dumps(evaluation_result, indent=2, ensure_ascii=False))

     MODUL 3: AUTOMATED BUSINESS RULE ENGINE      
{
  "nama_pendaftar": "BUDI SANTOSO",
  "nik": "3171012304980001",
  "usia_pendaftar": "28 Tahun",
  "pendidikan": "SMK TEKNIK MESIN",
  "status_akhir": "REJECTED",
  "log_catatan_sistem": [
    "REJECT: Usia pendaftar (28 tahun) di luar rentang syarat magang (18-26 tahun).",
    "INFO SYSTEM: Koreksi NIK: OCR membaca 17 digit (terdapat karakter '1' tambahan di awal). NIK dinormalisasi menjadi 16 digit.",
    "INFO SYSTEM: Koreksi Alamat: 'ILMERDEKANO12' dikoreksi menjadi 'JL MERDEKANO12'."
  ]
}
